# 07 - Final pipeline

Notebooks 05 and 06 end with two findings that decide this pipeline:

1. RT-DETR-l with default settings (`qb0`) is the best detector: +0.054 AP over YOLO11s
   (`af1`) at the same resolution and number of epochs, better on 24 of 26 clips.
2. The validation curves of both detectors are flat after a few epochs, so more passes over
   the same 1629 frames do not help. The limit is the data, not the training length.

Every step below changes one thing and is scored with the same evaluator on the same fold-0
holdout and `val` frames. Data and inference changes are kept apart from model changes.

| Step | Run | Change | Kind |
|---|---|---|---|
| 0 | qb0_rtdetr_l_1152 | best required model (notebook 06) | reference |
| 1 | p2_rtdetr_s5 | every 5th frame of folds 1-2 instead of every 10th (3243 frames), 15 epochs | data |
| 1b | p1_yolo11s_s5 | the same data change for YOLO11s (control) | data |
| 2 | `scripts/infer.py` | inference resolution, NMS on RT-DETR output, flip TTA, tiling | inference |
| 3 | `scripts/infer.py --ensemble` | weighted boxes fusion of several runs | inference |
| 4 | p3_rtdetr_s5_short | p2 with 5 epochs instead of 15, scored with last.pt | schedule |
| 5 | `scripts/temporal.py` | rescoring with the neighbouring frames of the clip | inference |
| 6 | f1_rtdetr_full_s5 | the qb3 recipe (notebook 06) on all 63 train and val clips, for the test set | final fit |

**Same number of iterations.** p1 and p2 see twice as many frames for half the epochs, and
the warm-up and the epochs without mosaic are halved too. The optimiser therefore makes the same
number of steps with the same learning-rate schedule as af1 and qb0, and only the diversity of
the frames changes.

Commands (all from the repository root):

```
python scripts/prepare_data.py --fold 0 --train-step 5
python scripts/train.py configs/pipeline/p2_rtdetr_s5.yaml configs/pipeline/p1_yolo11s_s5.yaml
python scripts/infer.py configs/query_based/qb0_rtdetr_l_1152.yaml --strategy tiles --merge nms
python scripts/infer.py --ensemble <result name> <result name> --weights 2 1
python scripts/train.py configs/pipeline/p3_rtdetr_s5_short.yaml
python scripts/temporal.py configs/query_based/qb0_rtdetr_l_1152.yaml --strategy tiles
python scripts/prepare_data.py --full --train-step 5
python scripts/train.py configs/pipeline/f1_rtdetr_full_s5.yaml
python scripts/infer.py configs/pipeline/f1_rtdetr_full_s5.yaml --strategy tiles --merge nms --test-only
```

**A caveat on fold 0.** `best.pt` is the epoch with the highest mAP on every 20th fold-0 frame,
so fold-0 numbers of a `best.pt` are slightly optimistic. `val` is never used for any choice
before this notebook and is the cleaner number.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.analysis import paired_comparison, per_clip_ap
from src.common import REPO_ROOT, RUNS_DIR
from src.predict import EVAL_SETS
from src.train_tools import load_config

INFERENCE_DIR = RUNS_DIR / "inference"
pd.set_option("display.precision", 4)

In [ ]:
def load_runs(family):
    found = {}
    for config_path in sorted((REPO_ROOT / "configs" / family).glob("*.yaml")):
        cfg = load_config(config_path)
        run_dir = RUNS_DIR / family / cfg["name"]
        if (run_dir / "metrics.json").exists():
            found[cfg["name"]] = {"cfg": cfg, "dir": run_dir,
                                  "metrics": json.loads((run_dir / "metrics.json").read_text())}
    return found


def load_inference_results():
    found = {}
    for path in sorted(INFERENCE_DIR.glob("*/metrics.json")):
        found[path.parent.name] = {"dir": path.parent, "metrics": json.loads(path.read_text())}
    return found


def metrics_table(entries):
    rows = []
    for name, entry in entries.items():
        fold0, val = entry["metrics"]["fold0"], entry["metrics"]["val"]
        rows.append({"run": name, "AP": fold0["AP"], "AP50": fold0["AP50"], "AP_small": fold0["AP_small"],
                     "val_AP": val["AP"], "val_AP50": val["AP50"], "val_AP_small": val["AP_small"],
                     "ms_per_image": val["ms_per_image"]})
    return pd.DataFrame(rows).set_index("run")


def clip_ap_of(detection_files):
    # detection_files maps eval set -> path of a COCO detections file
    ap = {}
    for eval_set, path in detection_files.items():
        detections = json.loads(path.read_text())
        for clip, value in per_clip_ap(EVAL_SETS[eval_set][1], detections).items():
            ap[f"{eval_set}/{clip}"] = value
    return ap


def compare(clip_ap, reference):
    table = paired_comparison(clip_ap, reference=reference).drop(index=reference)
    table["significant"] = table["mean_diff"].abs() >= 2 * table["std_err"]
    return table


families = ["anchor_free", "query_based", "pipeline"]
runs = {name: run for family in families for name, run in load_runs(family).items()}
inference = load_inference_results()
print("training runs:", list(runs))
print("inference results:", list(inference))

## 1. Data: more distinct frames at the same number of iterations

Each pipeline run is paired with the run it changes. `delta` is the change of the metric.

In [ ]:
data_pairs = {"p1_yolo11s_s5": "af1_imgsz1152", "p2_rtdetr_s5": "qb0_rtdetr_l_1152"}
data_pairs = {new: old for new, old in data_pairs.items() if new in runs and old in runs}

table = metrics_table(runs)
data_rows = []
for new, old in data_pairs.items():
    for metric in ["AP", "AP_small", "val_AP", "val_AP_small"]:
        data_rows.append({"run": new, "reference": old, "metric": metric, "reference_value": table.loc[old, metric],
                          "new_value": table.loc[new, metric], "delta": table.loc[new, metric] - table.loc[old, metric]})
pd.DataFrame(data_rows).set_index(["run", "metric"])

Validation mAP during training against the number of optimiser steps (epochs times frames per
epoch, in units of one pass over the 1629 frames of the step-10 set). Both runs of a pair end
at the same point.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for new, old in data_pairs.items():
    for name in (old, new):
        curve = pd.read_csv(runs[name]["dir"] / "results.csv")
        curve.columns = curve.columns.str.strip()
        n_frames = len(list((REPO_ROOT / Path(runs[name]["cfg"]["data"]).parent / "images" / "train").glob("*.jpg")))
        steps = curve["epoch"] * n_frames / 1629
        ax.plot(steps, curve["metrics/mAP50-95(B)"], "-" if name == new else "--", label=name)
ax.set_xlabel("training progress, passes over 1629 frames")
ax.set_ylabel("mAP@[.5:.95] (Ultralytics, fold 0)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
data_clip_ap = {name: clip_ap_of({s: runs[name]["dir"] / f"predictions_{s}.json" for s in EVAL_SETS})
                for pair in data_pairs.items() for name in pair}
for new, old in data_pairs.items():
    print(f"{new} against {old}")
    display(compare({new: data_clip_ap[new], old: data_clip_ap[old]}, reference=old))

**Finding.** Twice as many frames at the same number of iterations change nothing that is
larger than the noise, for either detector (YOLO11s: -0.005 +- 0.009 per clip, 13 clips better
and 13 worse; RT-DETR: +0.015 +- 0.011, 12 better and 14 worse). Neighbouring frames of a clip
are almost identical, so every 5th frame adds little that every 10th frame did not already show.
More data can only help through new clips, which is what the final fit adds.

## 2. Inference strategies on one trained model

All strategies run on the same checkpoint, so every difference comes from inference alone.
`ms_per_image` is the wall time of all passes and of the merging (batch 1, FP16).

| Strategy | What it does |
|---|---|
| plain | one pass on the full frame (as in notebooks 05 and 06) |
| nms | plain + class-wise NMS (IoU 0.7): removes duplicates the one-to-one matching left |
| plain at 1344 / 1536 | the model trained at 1152 px is run on a larger input |
| flip | full frame + mirrored frame, merged by NMS or by weighted boxes fusion (WBF) |
| tiles | full frame + 2x2 tiles with 20% overlap, each at 1152 px (about 1.7x zoom); boxes cut by an inner tile border are dropped (SAHI, arXiv:2202.06934) |
| tiles_flip | tiles + the mirrored full frame |
| tiles grid 3 | full frame + 3x3 tiles (about 2.4x zoom) |

In [ ]:
strategy_results = {name: r for name, r in inference.items() if "config" in r["metrics"]}
strategies = metrics_table(strategy_results)
strategies.insert(0, "model", [Path(r["metrics"]["config"]).stem for r in strategy_results.values()])
strategies.sort_values(["model", "AP"])

In [ ]:
# Paired per-clip comparison against the plain pass of the same model
for model_name, group in strategies.groupby("model"):
    config = strategy_results[group.index[0]]["metrics"]["config"]
    plain = f"{model_name}_plain_{load_config(REPO_ROOT / config)['imgsz']}"
    clip_ap = {name: clip_ap_of({s: inference[name]["dir"] / f"detections_{s}.json" for s in EVAL_SETS})
               for name in group.index}

    # Without a separate plain pass, the predictions of the training run are the plain pass
    if plain not in clip_ap and model_name in runs:
        clip_ap[plain] = clip_ap_of({s: runs[model_name]["dir"] / f"predictions_{s}.json" for s in EVAL_SETS})
    if plain not in clip_ap or len(clip_ap) < 2:
        continue
    print(f"{model_name}: against {plain}")
    display(compare(clip_ap, reference=plain))

**Finding.** Tiling with NMS is the only large gain: +0.021 +- 0.003 per-clip AP, better on all 26
clips, at 4.5 times the inference time. The zoomed tiles find small objects the full frame
misses (val AP_small 0.156 against 0.134). A larger input without tiles helps less (+0.008 at
1344 px, +0.010 at 1536 px), and flip TTA helps a little (+0.009). NMS on the plain RT-DETR
output changes nothing, as in notebook 06.

The gain carries over to the better model: tiles on qb3 give +0.020 +- 0.004 over its plain pass
(23 of 26 clips better), and qb3 with tiles is the best result on both sets (fold 0 0.325, val
0.287). Against qb0 with tiles it is still +0.009 +- 0.005, so the two gains add up.

More of the same does not help: adding the mirrored frame to the tiles (fold 0 0.318, val 0.271)
or a 3x3 grid (2.4x zoom; 0.315 / 0.264, twice the time) is not better than the 2x2 tiles
(0.317 / 0.275). At 2.4x zoom the tiles are small, and large objects are cut by more borders.

WBF is worse than NMS for both flip and tiles. WBF lowers the score of a box found by only some
of the parts, which is right for an ensemble of models that all see the whole frame, but here a
small object is often found only in its tile, so exactly the boxes tiling is meant to add are
pushed down. It also averages the boxes of a cluster, which merges neighbouring cars in dense
parking lots.

### Tiles and the frame width

On the public leaderboard tiles did not help the final model (f1: 0.2762 with tiles, 0.2792
without, section 8), although they gave +0.020 to +0.026 on our sets. The frame sizes explain it.

In [ ]:
from PIL import Image

from src.temporal import frames_table


def clip_widths(split):
    # All frames of a clip have the same size
    frames = frames_table(split)
    return {clip: Image.open(group["path"].iloc[0]).size[0] for clip, group in frames.groupby("clip")}


widths = {split: clip_widths(split) for split in ["train", "val", "test"]}
display(pd.DataFrame({split: pd.Series(w).value_counts() for split, w in widths.items()})
        .fillna(0).astype(int).rename_axis("frame width").sort_index())

QB3 = "qb3_bblr_nobiaswarm_1152"
tile_gain = pd.DataFrame({
    "plain": clip_ap_of({s: runs[QB3]["dir"] / f"predictions_{s}.json" for s in EVAL_SETS}),
    "tiles": clip_ap_of({s: inference[f"{QB3}_tiles_nms_1152"]["dir"] / f"detections_{s}.json" for s in EVAL_SETS}),
})
tile_gain["gain"] = tile_gain["tiles"] - tile_gain["plain"]
tile_gain["width"] = [widths[EVAL_SETS[key.split("/")[0]][0]][key.split("/")[1]] for key in tile_gain.index]
tile_gain.groupby("width")["gain"].agg(["mean", "count"])

**Finding.** 33 of the 48 test clips are at most 1360 px wide (down to 680 px), while most of our
training and evaluation clips are 1904 px or wider. Tiles help by recovering the pixels the
full-frame pass loses when the frame is scaled down to 1152 px, so their gain grows with the frame
width: on qb3 +0.006 per clip at 1344 px (only 4 clips), +0.019 at 1904 px and +0.067 at 3840 px.
A 1360 px frame loses much less in the full-frame pass, so the tiles have less to add. That they
turn negative on narrow test clips is our explanation, not a measurement: the 4 narrow clips in
these evaluation sets still gain a little. Notebook 09 scores tiles on all folds, which covers the
other narrow training clips as well.

Tiles are therefore used only for frames at least 1600 px wide (`--tile-min-width 1600`, 1296 of
the 3811 test frames); the other frames get the plain pass. The idea comes from the leaderboard
result above, so its public score is not an independent test of it. The threshold was set once
between the narrow test widths (at most 1360 px) and the common training width (1904 px) and was
not searched. On our sets it costs little because they have few narrow clips: qb3 fold 0
0.3246 to 0.3235, val 0.2872 to 0.2820 (`..._tiles_nms_minw1600_1152` in the table above).

## 3. Ensembles

Weighted boxes fusion (Solovyev et al., arXiv:1910.13302, implemented in `src/inference.py`):
boxes of the same class from different sources are clustered by IoU > 0.55, averaged with
their scores as weights, and the fused score is lowered when only some of the sources found
the object.

In [ ]:
ensemble_results = {name: r for name, r in inference.items() if "sources" in r["metrics"]}
if ensemble_results:
    ensembles = metrics_table(ensemble_results).drop(columns="ms_per_image")
    ensembles.insert(0, "weights", [str(r["metrics"]["weights"]) for r in ensemble_results.values()])
    display(ensembles.sort_values("AP"))

    # Paired comparison of every ensemble against its best source
    for name, result in ensemble_results.items():
        names = [name, *result["metrics"]["sources"]]
        clip_ap = {n: clip_ap_of({s: inference[n]["dir"] / f"detections_{s}.json" for s in EVAL_SETS}) for n in names}
        best_source = max(result["metrics"]["sources"], key=lambda n: inference[n]["metrics"]["val"]["AP"])
        print(f"{name}: against {best_source}")
        display(compare(clip_ap, reference=best_source))

**Finding.** Fusing RT-DETR with the best YOLO11s run does not help (-0.000 +- 0.003): YOLO11s
is worse on 24 of 26 clips, and its boxes add false positives about as often as missed objects.
Two RT-DETR runs with different recipes (qb3 and p3 of section 4, which are equally good) give
+0.002 +- 0.005 over p3 (16 clips better, 10 worse; val AP 0.279 against 0.282), again not
larger than the noise, at twice the inference time. The errors
of two RT-DETR-l runs from the same COCO weights are too similar for fusion to fix them.

## 4. Training schedule for the final fit

The fold-0 curves of qb0 and p2 peak after 4-8 epochs and then fall by about 0.01, and the
final fit has no holdout to choose the best epoch on. There are two ways out. p3 runs p2 for 5
epochs with a scaled warm-up and is scored with its last checkpoint, i.e. a short schedule that
ends near the peak. qb3 (notebook 06) is a recipe whose curve does not fall: it rises until
epoch 30, its last checkpoint is as good as its best, and it is the best model on val.

**Final fit f1**: the qb3 recipe on every 5th frame of all 63 train and val clips. 9 epochs of
5439 frames are the 30 x 1629 iterations of qb3, with the warm-up and the epochs without mosaic
scaled down in the same way. By section 1 the frame step does not matter at equal iterations,
so the only change against qb3 is the 21 new clips. f1 trains on the evaluation clips, so it is
not scored here; its test is the leaderboard (section 7).

In [ ]:
schedule_runs = [name for name in ["qb0_rtdetr_l_1152", "qb3_bblr_nobiaswarm_1152", "p2_rtdetr_s5", "p3_rtdetr_s5_short"] if name in runs]
display(metrics_table({name: runs[name] for name in schedule_runs}))

schedule_clip_ap = {name: clip_ap_of({s: runs[name]["dir"] / f"predictions_{s}.json" for s in EVAL_SETS})
                    for name in schedule_runs}
if "p3_rtdetr_s5_short" in schedule_clip_ap:
    for reference in ["qb0_rtdetr_l_1152", "qb3_bblr_nobiaswarm_1152"]:
        print("reference:", reference)
        display(compare(schedule_clip_ap, reference=reference))

**Finding.** Both ways out work. p3 (5 epochs, `last.pt`) is +0.014 +- 0.006 per clip over qb0
(19 of 26 clips better) and has the best plain-pass val AP (0.282), with a third of the
iterations of qb0 and qb3. Against qb3 it is +0.004 +- 0.005, i.e. the same within the noise: p3
is better on all 7 val clips and slightly worse on fold 0. The final fit uses the qb3 recipe,
because its curve was still rising at the end, so its `last.pt` does not depend on stopping at
the right epoch; with 21 more clips the peak of a short default schedule could move.

## 5. Temporal rescoring

The test frames are consecutive frames of 48 clips. `src/temporal.py` matches every box to the
best overlapping box of the same class in each of the `radius` frames before and after, and the
new score is the mean of its own score and the matched scores (mode `mean`: a frame without a
match counts as 0, so boxes that flicker lose score; mode `smooth`: only frames with a match
count). Radius 0 is the frame-by-frame result.

In [ ]:
from src.inference import from_coco, to_coco
from src.temporal import frames_table, neighbours, rescore

temporal_rows = []
for path in sorted((RUNS_DIR / "temporal").glob("*/metrics.json")):
    for row in json.loads(path.read_text()):
        temporal_rows.append({"result": path.parent.name, **row})
temporal = pd.DataFrame(temporal_rows)
if len(temporal):
    temporal[["iou_thr", "mode"]] = temporal[["iou_thr", "mode"]].fillna("-")
    temporal_ap = temporal.pivot_table(index=["result", "radius", "iou_thr", "mode"], columns="eval_set",
                                       values="AP").reset_index()
    baseline = temporal_ap[temporal_ap["radius"] == 0].set_index("result")
    for eval_set in EVAL_SETS:
        temporal_ap[f"delta_{eval_set}"] = temporal_ap[eval_set] - temporal_ap["result"].map(baseline[eval_set])
    display(temporal_ap.sort_values(["result", "radius"]))

In [ ]:
# Paired comparison of the best settings of the grid against radius 0
CANDIDATES = {"smooth r1": (1, 0.3, "smooth"), "smooth r2": (2, 0.3, "smooth"),
              "smooth r3": (3, 0.3, "smooth"), "mean r1": (1, 0.3, "mean")}


def rescored_clip_ap(result):
    ap = {name: {} for name in ["frame by frame", *CANDIDATES]}
    for eval_set, (split, gt_path) in EVAL_SETS.items():
        preds = from_coco(json.loads((RUNS_DIR / "temporal" / result / f"detections_{eval_set}.json").read_text()))
        centre_ids = [img["id"] for img in json.loads(gt_path.read_text())["images"]]
        window = neighbours(frames_table(split), centre_ids, 3)
        outputs = {"frame by frame": preds}
        for name, (radius, iou_thr, mode) in CANDIDATES.items():
            outputs[name] = rescore(preds, window, radius, iou_thr=iou_thr, mode=mode)
        for name, out in outputs.items():
            detections = [det for image_id in centre_ids if image_id in out for det in to_coco(out[image_id], image_id)]
            for clip, value in per_clip_ap(gt_path, detections).items():
                ap[name][f"{eval_set}/{clip}"] = value
    return ap


if len(temporal):
    for result in temporal_ap["result"].unique():
        print(result)
        display(compare(rescored_clip_ap(result), reference="frame by frame"))

**Finding.** Mode `smooth` with IoU 0.3 helps at every radius, mode `mean` does not. On qb0 with
tiles, `smooth` at radius 2 gives +0.006 +- 0.002 per clip (23 of 26 clips better), radius 1 and
3 give the same within the noise. On the plain pass of qb0 the gain is half as large (+0.003 +-
0.001): tiling adds many low-score boxes of small objects, and these are the boxes whose score
depends most on whether the object is also seen in the neighbouring frames. On the plain pass of
qb3 it is +0.005 +- 0.001 (20 of 26 clips better), so the gain holds for the final recipe too.

`mean` punishes every frame without a match. A small object is often missed in one of the
neighbouring frames, so `mean` lowers the score of true detections almost as often as of false
ones, and the larger the radius, the worse it gets. IoU 0.5 is too strict for small fast-moving
objects, whose boxes of two frames overlap less. Radius 2 is used for the test: it is in the
middle of the settings that work, so the choice is not tuned to one evaluation set.

## 6. Summary of all models

Best run of every family on the same evaluation sets. The anchor-based numbers come from the
torchvision notebooks and are evaluated with the same `src/evaluate.py`.

In [ ]:
anchor_based = {
    # "faster_rcnn_tuned": {"AP": ..., "AP50": ..., "AP_small": ..., "val_AP": ..., "ms_per_image": ...},
    # "retinanet_tuned": {...},
}
training = metrics_table(runs)
training.insert(0, "kind", [runs[name]["cfg"]["family"] for name in training.index])

# A plain pass at the training size repeats the training run, so only the other strategies are added
def repeats_training_run(result):
    metrics = result["metrics"]
    return metrics["strategy"] == "plain" and metrics["imgsz"] == load_config(REPO_ROOT / metrics["config"])["imgsz"]

extra = strategies.drop(columns="model").loc[[name for name, r in strategy_results.items()
                                              if not repeats_training_run(r)]]
extra.insert(0, "kind", "inference")
anchor_table = pd.DataFrame(anchor_based).T
anchor_table.insert(0, "kind", "anchor_based")
summary = pd.concat([anchor_table, training, extra])
summary.sort_values("val_AP")

## 7. Final pipeline

Every component is kept only if its paired per-clip gain is larger than twice its standard error:

| Component | Choice | Evidence |
|---|---|---|
| detector | RT-DETR-l, COCO weights, 1152 px | +0.054 over YOLO11s (notebook 06) |
| training recipe | lr 1e-4, backbone lr 0.1x, weight decay 1e-4, no bias warm-up | qb3: +0.011 over qb0, curve does not fall (notebook 06) |
| data | all 63 train and val clips, every 5th frame | new clips; frame step alone does not matter (section 1) |
| schedule | 30 x 1629 iterations, last checkpoint | qb3 curve rises until the last epoch (section 4) |
| inference | full frame + 2x2 tiles, NMS, only for frames at least 1600 px wide | +0.020 on qb3; frame widths of the test set (section 2) |
| post-processing | temporal rescoring, `smooth`, radius 2, IoU 0.3 | +0.006 on qb0 with tiles (section 5) |

Rejected: larger input without tiles, flip TTA, WBF, an ensemble with YOLO11s, mode `mean`,
500 queries (notebook 06).

Reproduce from the repository root:

```
python scripts/prepare_data.py --full --train-step 5
python scripts/train.py configs/pipeline/f1_rtdetr_full_s5.yaml
python scripts/temporal.py configs/pipeline/f1_rtdetr_full_s5.yaml --strategy tiles --tile-min-width 1600 --test-only --radius 2 --iou 0.3 --mode smooth
```

The last command writes `submissions/f1_rtdetr_full_s5_tiles_minw1600_1152_temporal_smooth_r2_iou0.3.csv`.

## 8. Submissions

Public leaderboard scores of the submitted files, filled in after every submission.

In [ ]:
public_lb = {
    "af1_imgsz1152_1152": 0.19384,
    "qb3_bblr_nobiaswarm_1152_tiles_nms_1152": 0.26131,
    "qb3_bblr_nobiaswarm_1152_tiles_1152_temporal_smooth_r2_iou0.3": 0.27228,
    "f1_rtdetr_full_s5_plain_1152": 0.27921,
    "f1_rtdetr_full_s5_tiles_nms_1152": 0.27619,
    "f1_rtdetr_full_s5_tiles_1152_temporal_smooth_r2_iou0.3": 0.28161,
}
files = sorted((REPO_ROOT / "submissions").glob("*.csv"))
pd.DataFrame({"file": [f.name for f in files], "public_lb": [public_lb.get(f.stem) for f in files]})